In [3]:
## 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [4]:
## 2. Read and prepare data
# Read dataset
df = pd.read_csv("prep.csv")

# One-hot encoding
df = pd.get_dummies(df, drop_first=True)

# Split features and target
X = df.drop("classification_yes", axis=1)
y = df["classification_yes"]

In [5]:
## 3. Load all classifier functions

def split_and_scale(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=0
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test


def evaluate_accuracy(X, y):
    X_train, X_test, y_train, y_test = split_and_scale(X, y)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

In [6]:
## 4. Test for Backward Elimination

features = list(X.columns)
results = []

current_X = X.copy()
current_score = evaluate_accuracy(current_X, y)

results.append((len(features), current_score, features.copy()))

while len(features) > 1:
    scores_after_removal = []

    for feature in features:
        temp_features = features.copy()
        temp_features.remove(feature)

        temp_X = X[temp_features]
        score = evaluate_accuracy(temp_X, y)

        scores_after_removal.append((feature, score))

    # Remove the feature whose removal gives BEST accuracy
    feature_to_remove, best_score = max(scores_after_removal, key=lambda x: x[1])

    if best_score >= current_score:
        features.remove(feature_to_remove)
        current_score = best_score
        current_X = X[features]

        results.append((len(features), current_score, features.copy()))
    else:
        break

In [7]:
## 5 Create the result dataframe and display Best Feature
results_df = pd.DataFrame(
    [(r[0], r[1]) for r in results],
    columns=["Num_features", "Accuracy"])

best_row = results_df.loc[results_df["Accuracy"].idxmax()]
best_feature_count = int(best_row["Num_features"])

best_features = results[results_df["Accuracy"].idxmax()][2]

print("Best number of features:", best_feature_count)


Best number of features: 26


In [9]:
##  6.Print the selected features
print("Selected Features (Backward Elimination):")
for f in best_features:
    print(f)

Selected Features (Backward Elimination):
age
bp
al
su
bgr
bu
sc
sod
pot
hrmo
pcv
wc
rc
sg_b
sg_d
sg_e
rbc_normal
pc_normal
pcc_present
ba_present
htn_yes
dm_yes
cad_yes
appet_yes
pe_yes
ane_yes
